06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [38]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch

# WORKING WITH 
datasetPath = 'data/clothDataset_5_.csv'

cloth_info = pd.read_csv(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
#print('cloth_info: \n{}'.format(cloth_info))
print(cloth_info.filter(like='md').iloc[0]) # TODO PAU MIRALO
#print(cloth_info.filter(like='u').iloc[0]) # PROBABLEMENTE CORRECTO
#print(cloth_info.filter(regex=r'^v\d+').iloc[0]) # PROBABLEMENTE CORRECTO

cloth_info shape: (42, 326)
md0     3.402823e+38
md1     3.402823e+38
md2     3.402823e+38
md3     3.402823e+38
md4     3.402823e+38
md5     3.402823e+38
md6     3.402823e+38
md7     3.402823e+38
md8     3.402823e+38
md9     3.402823e+38
md10    3.402823e+38
md11    3.402823e+38
md12    3.402823e+38
md13    3.402823e+38
md14    0.000000e+00
md15    0.000000e+00
md16    3.402823e+38
md17    3.402823e+38
md18    3.402823e+38
md19    3.402823e+38
md20    0.000000e+00
md21    3.402823e+38
md22    3.402823e+38
md23    0.000000e+00
md24    0.000000e+00
Name: 0, dtype: float64


In [ ]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=25):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        if isinstance(csv_data, str) and "frame,x0" in csv_data:
            self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        else:
            self.data = pd.read_csv(csv_data)
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']

        self.position_prefixes = ['x', 'y', 'z']
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')

        # Quitamos primera fila de outputs (no es el output de nada) y ultima fila de input (no tiene output)
        self.output_positions = self.output_positions.iloc[1:]
        self.data = self.data.iloc[:-1]
        

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)
    
    def num_features(self):
        # Number of columns in a row (pos, vel, sdf, uv per vertex)
        return len(self.feature_prefixes)
    
    def num_vertex(self):
        return self.num_vertices
    
    def _get_frame_output_tensor(self, idx):
         row = self.output_positions.iloc[idx]
         frame_data = []
         for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.position_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

         tensor_data = torch.tensor(np.array(frame_data))

         return tensor_data
    
    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_output_tensor(idx) # +1 ya no TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(datasetPath)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break

Batch Shape: torch.Size([4, 25, 13])
tensor([[[ 5.1037e+01, -3.0685e+01,  1.9180e+01,  ...,  3.4028e+38,
           7.5000e-01,  0.0000e+00],
         [ 4.0647e+01, -1.0436e+01,  4.5809e+01,  ...,  3.4028e+38,
           1.0000e+00,  2.5000e-01],
         [ 6.3007e+01, -1.8200e+01,  3.7230e+01,  ...,  3.4028e+38,
           1.0000e+00,  0.0000e+00],
         ...,
         [ 7.0538e+00,  2.7308e+01, -4.9896e+01,  ...,  3.4028e+38,
           0.0000e+00,  7.5000e-01],
         [ 1.4000e-01,  5.1330e+01, -2.5139e+01,  ...,  0.0000e+00,
           2.5000e-01,  1.0000e+00],
         [ 1.3999e-01,  5.1330e+01, -5.0139e+01,  ...,  0.0000e+00,
           0.0000e+00,  1.0000e+00]],

        [[-2.5163e+01, -4.4499e+01,  2.0731e+01,  ...,  3.4028e+38,
           7.5000e-01,  0.0000e+00],
         [-2.8445e+01, -1.7484e+01,  4.3604e+01,  ...,  3.4028e+38,
           1.0000e+00,  2.5000e-01],
         [-3.8458e+01, -4.0566e+01,  4.1537e+01,  ...,  3.4028e+38,
           1.0000e+00,  0.0000e+00],
  

In [61]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        #print(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas

# nn.Linear in PyTorch is designed to handle 3D tensors seamlessly.  
# When a 3D input tensor (e.g., batch_size, sequence_length, features) is provided,
#  the layer applies the linear transformation only to the last dimension (the features dimension),
#  preserving all other dimensions.

model = MyModule(num_inputs=13, num_hidden= 2, num_outputs=3)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
    for batch_t, batch_t1 in dataloader:
        pred = model(batch_t)
        loss = criterion(pred, batch_t1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print(str(loss.item()))
        print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: inf
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 1, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 2, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 3, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 4, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Loss: nan
Epoch 5, Los

In [ ]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
